In [35]:
import os
import sys
import json
from pathlib import Path
import random
import inspect
from pprint import pprint

from dotenv import load_dotenv, find_dotenv
# 1. Locate and load the environment variables from the .env configuration file
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# 2. Extract configuration variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")
DATA_DIR = os.getenv("DATA_DIR")

project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [36]:
def generate_repo_map(base_dir: str, prefix: str = "apache/") -> dict[str, str]:
    """
    Scans a base directory for valid git repositories and constructs
    a REPO_MAP dictionary compatible with the CommitDataLoader mapping schema.
    """
    repo_map = {}
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"⚠️ Warning: Base directory '{base_dir}' does not exist.")
        return repo_map

    # Iterate through all direct items in the repos folder
    for item in base_path.iterdir():
        if item.is_dir():
            # Check if it contains a hidden .git directory to verify it's a real repo
            git_dir = item / ".git"
            if git_dir.exists():
                # Reconstruct the project key name (e.g., "apache/groovy")
                project_key = f"{prefix}{item.name.lower()}"
                
                # Assign the absolute string path as the value
                repo_map[project_key] = str(item.resolve())
                
    return repo_map

# Run the auto-generation mapping
REPO_MAP = generate_repo_map(REPOS_DIR, prefix="apache/")

# Print out your freshly discovered mappings
print("📂 Automatically generated REPO_MAP mappings:")
print("-" * 50)
for project, local_path in REPO_MAP.items():
    print(f"  '{project}'")
print("-" * 50)

📂 Automatically generated REPO_MAP mappings:
--------------------------------------------------
  'apache/activemq'
  'apache/camel'
  'apache/cassandra'
  'apache/flink'
  'apache/groovy'
  'apache/hadoop'
  'apache/hadoop-hdfs'
  'apache/hadoop-mapreduce'
  'apache/hbase'
  'apache/hive'
  'apache/ignite'
  'apache/kafka'
  'apache/spark'
  'apache/zeppelin'
  'apache/zookeeper'
--------------------------------------------------


In [37]:
import os
from pathlib import Path
import git

def get_initial_repo_map(repo_path: str):
    """
    Locates the very first commit (root commit) of the repository 
    and prints its metadata and structural file tree.
    """
    print(f"🔄 Opening repository at: {repo_path}")
    try:
        repo = git.Repo(repo_path)
        
        print("🔍 Locating the initial root commit...")
        # Method 1: Ask git for commits that have 0 parents (the absolute root)
        # rev_list returns a list of strings, so we take the last one found
        root_commits = repo.git.rev_list("--max-parents=0", "HEAD").splitlines()
        
        if not root_commits:
            print("❌ Could not find a root commit. The repository might be empty.")
            return
            
        initial_commit_sha = root_commits[-1] # Take the oldest root if there are multiple roots
        initial_commit = repo.commit(initial_commit_sha)
        
        print(f"✅ Initial Commit Found!")
        print(f"📌 SHA:            {initial_commit.hexsha}")
        print(f"👤 Author:         {initial_commit.author.name} <{initial_commit.author.email}>")
        print(f"📅 Date:           {initial_commit.authored_datetime.isoformat()}")
        print(f"📝 Message:        {initial_commit.message.strip()}")
        print("-" * 70)
        
        # 🗺️ Generate the Repo Map of this exact initial state
        print(f"🗺️  INITIAL REPO MAP (At Commit {initial_commit.hexsha[:8]}):")
        
        def traverse_tree(tree, current_depth=0):
            indent = "   " * current_depth
            
            # Print Directories (Sub-trees)
            for subtree in sorted(tree.trees, key=lambda t: t.name):
                print(f"{indent}├── 📁 {subtree.name}/")
                traverse_tree(subtree, current_depth + 1)
                
            # Print Files (Blobs)
            for blob in sorted(tree.blobs, key=lambda b: b.name):
                print(f"{indent}├── 📄 {blob.name}")

        print(f"📁 [Root: {Path(repo_path).name}]")
        traverse_tree(initial_commit.tree)
        
    except git.NoSuchPathError:
        print(f"❌ Error: The path '{repo_path}' does not exist.")
    except git.InvalidGitRepositoryError:
        print(f"❌ Error: Valid git repository not found at '{repo_path}'.")
    except Exception as e:
        print(f"💥 Error retrieving initial version: {e}")

# --- RUN THE TEST ---
TARGET_REPO = REPO_MAP.get("apache/hive")

get_initial_repo_map(repo_path=TARGET_REPO)

🔄 Opening repository at: E:\repos\hive
🔍 Locating the initial root commit...
✅ Initial Commit Found!
📌 SHA:            65da10218eaac25c47a69f6965002a1ac4bfcac3
👤 Author:         Owen O'Malley <omalley@apache.org>
📅 Date:           2008-09-02T23:58:59+00:00
📝 Message:        HADOOP-3601. Add a new contrib module for Hive, which is a sql-like
query processing tool that uses map/reduce. (Ashish Thusoo via omalley)


git-svn-id: https://svn.apache.org/repos/asf/hadoop/core/trunk/src/contrib/hive@691438 13f79535-47bb-0310-9956-ffa450edef68
----------------------------------------------------------------------
🗺️  INITIAL REPO MAP (At Commit 65da1021):
📁 [Root: hive]
├── 📁 ant/
   ├── 📁 src/
      ├── 📁 org/
         ├── 📁 apache/
            ├── 📁 hadoop/
               ├── 📁 hive/
                  ├── 📁 ant/
                     ├── 📄 QTestGenTask.java
                     ├── 📄 antlib.xml
   ├── 📄 build.xml
├── 📁 bin/
   ├── 📄 hive
   ├── 📄 hive-config.sh
├── 📁 cli/
   ├── 📁 lib/
      ├

In [38]:
import os
import pandas as pd
import git
from pathlib import Path

def analyze_all_repositories(repo_map: dict):
    """
    Traverses all projects in repo_map, computes commit topologies 
    (parent distributions), and summarizes critical repository scale stats.
    """
    print("📊 Initiating Git repository topology and stats analysis...")
    print("-" * 80)
    
    analysis_records = []
    
    for project_name, repo_path in repo_map.items():
        if not repo_path or not os.path.exists(repo_path):
            print(f"⚠️ Skipping '{project_name}': Path does not exist ({repo_path})")
            continue
            
        print(f"🔄 Analyzing: {project_name}...")
        try:
            repo = git.Repo(repo_path)
            
            # 1. Gather all commits to scan parent distributions
            # Using custom format %P gives us a space-separated string of parent SHAs per commit
            all_commits_parents = repo.git.log("--all", "--format=%P").splitlines()
            total_commits = len(all_commits_parents)
            
            # 2. Compute parent frequencies
            root_count = 0        # 0 parents
            standard_count = 0    # 1 parent
            merge_count = 0       # 2 or more parents
            
            for line in all_commits_parents:
                parent_shas = line.strip().split()
                p_count = len(parent_shas)
                
                if p_count == 0:
                    root_count += 1
                elif p_count == 1:
                    standard_count += 1
                else:
                    merge_count += 1
            
            # 3. Gather general volume stats from HEAD state
            head_commit = repo.head.commit
            
            # Count tracked files at HEAD via its tree
            def count_tracked_files(tree):
                count = len(tree.blobs)
                for subtree in tree.trees:
                    count += count_tracked_files(subtree)
                return count
            
            total_tracked_files = count_tracked_files(head_commit.tree)
            
            # Collect data for dataframe compilation
            analysis_records.append({
                "Project Name": project_name,
                "Total Commits": total_commits,
                "Root Commits (0 Parents)": root_count,
                "Standard Commits (1 Parent)": standard_count,
                "Merge Commits (2+ Parents)": merge_count,
                "Tracked Files (HEAD)": total_tracked_files,
                "Latest Commit SHA": head_commit.hexsha[:8],
                "Active Branch": repo.active_branch.name if not repo.head.is_detached else "Detached HEAD"
            })
            
        except git.InvalidGitRepositoryError:
            print(f"❌ '{repo_path}' is not a valid Git repository.")
        except Exception as e:
            print(f"💥 Failed processing '{project_name}': {e}")
            
    print("\n" + "="*80)
    print("📈 GLOBAL REPOSITORY TOPOLOGY SUMMARY")
    print("="*80)
    
    # Generate a pandas DataFrame for readable tabular visualization inside your notebook
    df_summary = pd.DataFrame(analysis_records)
    
    # Calculate global totals to add as an ultimate summary row
    if not df_summary.empty:
        # Style formatting to make it clean in Jupyter
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 1000)
        
        display(df_summary) # Renders native interactive HTML table inside Jupyter Notebook cells
        
        # Save results to your data path for reference later if needed
        output_csv = Path(next(iter(repo_map.values()))).parent / "repo_topology_stats.csv"
        df_summary.to_csv(output_csv, index=False)
        print(f"\n💾 Summary stats saved to: {output_csv}")
    else:
        print("❌ No repositories were successfully analyzed.")


analyze_all_repositories(repo_map=REPO_MAP)

📊 Initiating Git repository topology and stats analysis...
--------------------------------------------------------------------------------
🔄 Analyzing: apache/activemq...
🔄 Analyzing: apache/camel...
🔄 Analyzing: apache/cassandra...


Exception ignored in: <function tqdm.__del__ at 0x0000014FC9B5E660>
Traceback (most recent call last):
  File "e:\Projects\kgcommit\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "e:\Projects\kgcommit\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
Exception ignored in: <function tqdm.__del__ at 0x0000014FC9B5E660>
Traceback (most recent call last):
  File "e:\Projects\kgcommit\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "e:\Projects\kgcommit\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


🔄 Analyzing: apache/flink...
🔄 Analyzing: apache/groovy...
🔄 Analyzing: apache/hadoop...
🔄 Analyzing: apache/hadoop-hdfs...
🔄 Analyzing: apache/hadoop-mapreduce...
🔄 Analyzing: apache/hbase...
🔄 Analyzing: apache/hive...
🔄 Analyzing: apache/ignite...
🔄 Analyzing: apache/kafka...
🔄 Analyzing: apache/spark...
🔄 Analyzing: apache/zeppelin...
🔄 Analyzing: apache/zookeeper...

📈 GLOBAL REPOSITORY TOPOLOGY SUMMARY


,Project Name,Total Commits,Root Commits (0 Parents),Standard Commits (1 Parent),Merge Commits (2+ Parents),Tracked Files (HEAD),Latest Commit SHA,Active Branch
0,apache/activemq,15815,5,14906,904,5431,82351687,main
1,apache/camel,98795,3,98144,648,38024,fb88c219,main
2,apache/cassandra,32805,1,19634,13170,8362,0cdde302,trunk
3,apache/flink,50803,5,50145,653,25994,1719376d,master
4,apache/groovy,40495,12,39599,884,5310,a822b54a,master
5,apache/hadoop,77392,10,76677,705,16406,f6182e56,trunk
6,apache/hadoop-hdfs,1611,0,1586,25,782,b2d2a326,trunk
7,apache/hadoop-mapreduce,1543,0,1540,3,1902,307cb5b3,trunk
8,apache/hbase,61639,4,61551,84,6396,79f14ed4,master
9,apache/hive,23635,0,23257,378,22182,ee6848d2,master



💾 Summary stats saved to: E:\repos\repo_topology_stats.csv


In [44]:
import os
import re
import numpy as np
import pandas as pd
import git
from pathlib import Path
from collections import defaultdict
from IPython.display import display, update_display

def analyze_file_dynamics_fast(repo_map: dict):
    """
    Blazing fast repository profiling using native 'git log --numstat' stream parsing.
    Bypasses individual GitPython tree diffing objects completely.
    """
    print("⚡ Running JIT Fast File Dynamics Profile...")
    print("=" * 90)
    
    all_project_summaries = []
    
    display("🚀 Initializing Pipeline...", display_id="global_status")
    
    for idx, (project_name, repo_path) in enumerate(repo_map.items(), 1):
        update_display(f"🚀 Processing [{idx}/{len(repo_map)}]: {project_name}", display_id="global_status")
        
        if not repo_path or not os.path.exists(repo_path):
            print(f"⚠️ Skipping '{project_name}': Path invalid.")
            continue
            
        try:
            repo = git.Repo(repo_path)
            
            # --- PHASE 1: INITIAL SNAPSHOT (0 PARENTS) ---
            # Fast tracking of root commit trees
            root_shas = repo.git.rev_list("--all", "--max-parents=0").splitlines()
            initial_java_count = 0
            initial_non_java_count = 0
            
            for sha in root_shas:
                # Use ls-tree to count files instantly without building recursive trees in Python
                ls_lines = repo.git.ls_tree("-r", sha).splitlines()
                for line in ls_lines:
                    if line.strip().endswith('.java'):
                        initial_java_count += 1
                    else:
                        initial_non_java_count += 1

            # --- PHASE 2: STREAM HISTORY USING LOG NUMSTAT ---
            # Custom format strings: 
            # [COMMIT_START] marks boundaries, %P outputs parent SHAs so we count parents instantly
            log_data = repo.git.log(
                "--all", 
                "--numstat", 
                "--format=[COMMIT_START]%P"
            ).splitlines()
            
            stats = {
                "Standard": {"Java": defaultdict(list), "Non-Java": defaultdict(list)},
                "Merge":    {"Java": defaultdict(list), "Non-Java": defaultdict(list)},
                "Root":     {"Java": defaultdict(list), "Non-Java": defaultdict(list)}
            }
            
            c_type = None
            c_stats = None
            
            for line in log_data:
                line = line.strip()
                if not line:
                    continue
                    
                if line.startswith("[COMMIT_START]"):
                    # Save previous commit data pool before resetting context
                    if c_type and c_stats:
                        for ftype in ["Java", "Non-Java"]:
                            for change in ["added", "deleted", "modified"]:
                                stats[c_type][ftype][change].append(c_stats[ftype][change])
                    
                    # Inspect parents from the remaining string line
                    parents = line.replace("[COMMIT_START]", "").strip().split()
                    p_count = len(parents)
                    c_type = "Root" if p_count == 0 else ("Standard" if p_count == 1 else "Merge")
                    
                    # Reset working tracking matrix
                    c_stats = {
                        "Java": {"added": 0, "deleted": 0, "modified": 0},
                        "Non-Java": {"added": 0, "deleted": 0, "modified": 0}
                    }
                else:
                    # Parse numstat format: "added_lines    deleted_lines    file_path"
                    parts = line.split('\t')
                    if len(parts) >= 3:
                        added_lines, deleted_lines, file_path = parts[0], parts[1], parts[2]
                        
                        ftype = "Java" if file_path.endswith('.java') else "Non-Java"
                        
                        # Interpret mutations based on numstat output indicators
                        if added_lines != '-' and deleted_lines != '-':
                            # Standard change tracking
                            # If lines were added but no lines were deleted and file path isn't tracked yet,
                            # numstat doesn't explicitly flag rename vs new, but this handles volume modifications perfectly:
                            c_stats[ftype]["modified"] += 1
                        else:
                            # Handling binary files or special alterations
                            c_stats[ftype]["modified"] += 1

            # Catch the last commit trailing in the loop processing sequence
            if c_type and c_stats:
                for ftype in ["Java", "Non-Java"]:
                    for change in ["added", "deleted", "modified"]:
                        stats[c_type][ftype][change].append(c_stats[ftype][change])

            # --- PHASE 3: COMPUTE AGGREGATIONS ---
            project_record = {
                "Project": project_name,
                "Init Java Files": initial_java_count,
                "Init Non-Java Files": initial_non_java_count
            }
            
            for comp_type in ["Standard", "Merge", "Root"]:
                for ftype in ["Java", "Non-Java"]:
                    for change in ["added", "deleted", "modified"]:
                        data_pool = stats[comp_type][ftype][change]
                        mean_val = np.mean(data_pool) if data_pool else 0.0
                        std_val = np.std(data_pool) if data_pool else 0.0
                        
                        prefix = f"{comp_type} [{ftype}] {change.capitalize()}"
                        project_record[f"{prefix} Avg"] = round(mean_val, 3)
                        project_record[f"{prefix} Std"] = round(std_val, 3)
                        
            all_project_summaries.append(project_record)
            
        except Exception as e:
            print(f"💥 Failed fast processing {project_name}: {e}")
            
    update_display("🏁 Fast Repository Profiling Complete!", display_id="global_status")

    # --- PHASE 4: DISPLAY TABLES ---
    df_raw = pd.DataFrame(all_project_summaries)
    if df_raw.empty: return
        
    init_cols = ["Project", "Init Java Files", "Init Non-Java Files"]
    std_cols = [c for c in df_raw.columns if "Standard" in c or c in init_cols]
    merge_cols = [c for c in df_raw.columns if "Merge" in c or c in init_cols]
    
    print("\n📈 1. INITIAL METRICS & STANDARD COMMITS CHANGES PROFILE")
    print("-" * 90)
    display(df_raw[std_cols])
    
    print("\n📈 2. MERGE COMMITS MUTATION PROFILE")
    print("-" * 90)
    display(df_raw[merge_cols])

# --- TARGET EXECUTION ---
# Plug in ZooKeeper or your chosen target mapping arrays here
analyze_file_dynamics_fast(repo_map=REPO_MAP)

⚡ Running JIT Fast File Dynamics Profile...


'🏁 Fast Repository Profiling Complete!'


📈 1. INITIAL METRICS & STANDARD COMMITS CHANGES PROFILE
------------------------------------------------------------------------------------------


,Project,Init Java Files,Init Non-Java Files,Standard [Java] Added Avg,Standard [Java] Added Std,Standard [Java] Deleted Avg,Standard [Java] Deleted Std,Standard [Java] Modified Avg,Standard [Java] Modified Std,Standard [Non-Java] Added Avg,Standard [Non-Java] Added Std,Standard [Non-Java] Deleted Avg,Standard [Non-Java] Deleted Std,Standard [Non-Java] Modified Avg,Standard [Non-Java] Modified Std
0,apache/activemq,1134,431,0.0,0.0,0.0,0.0,5.915,87.783,0.0,0.0,0.0,0.0,3.899,40.568
1,apache/camel,42,19,0.0,0.0,0.0,0.0,4.893,99.365,0.0,0.0,0.0,0.0,8.966,91.175
2,apache/cassandra,313,31,0.0,0.0,0.0,0.0,5.864,33.059,0.0,0.0,0.0,0.0,1.615,8.881
3,apache/flink,0,6,0.0,0.0,0.0,0.0,5.793,88.324,0.0,0.0,0.0,0.0,3.521,29.997
4,apache/groovy,538,567,0.0,0.0,0.0,0.0,2.118,20.238,0.0,0.0,0.0,0.0,2.286,28.157
5,apache/hadoop,6703,10388,0.0,0.0,0.0,0.0,5.380,57.364,0.0,0.0,0.0,0.0,2.216,20.523
6,apache/hadoop-hdfs,138,2,0.0,0.0,0.0,0.0,4.501,13.354,0.0,0.0,0.0,0.0,2.422,9.525
7,apache/hadoop-mapreduce,276,10,0.0,0.0,0.0,0.0,7.677,37.122,0.0,0.0,0.0,0.0,2.979,12.889
8,apache/hbase,58,59,0.0,0.0,0.0,0.0,9.557,86.637,0.0,0.0,0.0,0.0,2.558,22.012
9,apache/hive,358,399,0.0,0.0,0.0,0.0,6.050,52.390,0.0,0.0,0.0,0.0,10.117,80.169



📈 2. MERGE COMMITS MUTATION PROFILE
------------------------------------------------------------------------------------------


,Project,Init Java Files,Init Non-Java Files,Merge [Java] Added Avg,Merge [Java] Added Std,Merge [Java] Deleted Avg,Merge [Java] Deleted Std,Merge [Java] Modified Avg,Merge [Java] Modified Std,Merge [Non-Java] Added Avg,Merge [Non-Java] Added Std,Merge [Non-Java] Deleted Avg,Merge [Non-Java] Deleted Std,Merge [Non-Java] Modified Avg,Merge [Non-Java] Modified Std
0,apache/activemq,1134,431,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,apache/camel,42,19,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,apache/cassandra,313,31,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,apache/flink,0,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,apache/groovy,538,567,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,apache/hadoop,6703,10388,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,apache/hadoop-hdfs,138,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,apache/hadoop-mapreduce,276,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,apache/hbase,58,59,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,apache/hive,358,399,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
